In [1]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from sqlalchemy import create_engine, Integer, String, Date, Boolean, ForeignKey, Numeric
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from typing import List, Optional
from datetime import date

In [3]:
engine = create_engine("sqlite:///bandhub.db")

class Base(DeclarativeBase):
    ...


In [ ]:
class Employee(Base):
    __tablename__ = "employee"
    
    id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(String(64), nullable=False)
    surname: Mapped[str] = mapped_column(String(64), nullable=False)
    national_id: Mapped[str] = mapped_column(String(20), nullable=False, unique=True)
    role: Mapped[str] = mapped_column(String(20), nullable=False)

    # Relació 1:1
    credentials: Mapped[Optional["AppCredential"]] = relationship(back_populates="employee", uselist=False)


class AppCredential(Base):
    __tablename__ = "app_credential"
    

    id_employee: Mapped[int] = mapped_column(Integer, ForeignKey("employee.id", ondelete="CASCADE"), primary_key=True)
    
    email: Mapped[str] = mapped_column(String(128), unique=True, nullable=False)
    password_hash: Mapped[str] = mapped_column(String(255), nullable=False)

    # Relació inversa
    employee: Mapped["Employee"] = relationship(back_populates="credentials")

In [ ]:
class Category(Base):
    __tablename__ = "category"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    category_name: Mapped[str] = mapped_column(String(64), nullable=False)

    # Relació: Una categoria té una LLISTA de plats
    dishes: Mapped[List["Dish"]] = relationship(back_populates="category")

class Dish(Base):
    __tablename__ = "dish"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    dish_name: Mapped[str] = mapped_column(String(64), nullable=False)
    description: Mapped[Optional[str]] = mapped_column(String)
    is_active: Mapped[bool] = mapped_column(default=True)
    
    # La Foreign Key sempre va al costat "Many"
    id_category: Mapped[int] = mapped_column(ForeignKey("category.id"), nullable=False)
    
    # Relació: Un plat pertany a UNA sola categoria
    category: Mapped["Category"] = relationship(back_populates="dishes")